In [0]:
storage_account_name = "cloudproject60306117"
storage_account_key = "AdxIWpu+RCT/r8Gr8rvVZxi25XMS/T0QnFugMQMzWXp6P7RHljEIpEZooyARZ3VJAEalcNRoGaDu+AStblz2Bw=="

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

In [0]:
from pyspark.sql.functions import col, hour, dayofweek, when

# Load cleaned trips from Silver layer
clean_path = "abfss://processed@cloudproject60306117.dfs.core.windows.net/clean_trips/"
df = spark.read.parquet(clean_path)

# Enrich with time-based features
df = df.withColumn("pickup_hour", hour(col("tpep_pickup_datetime")))
df = df.withColumn("pickup_day_of_week", dayofweek(col("tpep_pickup_datetime")))

# Derived feature: fare efficiency 
df = df.withColumn(
    "fare_per_mile",
    when(col("trip_distance") > 0, col("fare_amount") / col("trip_distance")).otherwise(0)
)

# Save enriched dataset back to Silver
enriched_path = "abfss://processed@cloudproject60306117.dfs.core.windows.net/enriched_trips/"
df.write.mode("overwrite").parquet(enriched_path)

print("Enriched dataset rows:", df.count())


Enriched dataset rows: 12193830
